In [ ]:
import torch


In [ ]:
import torch.nn as nn


In [ ]:
import tiktoken
tokenizer = tiktoken.get_encoding("gpt2")

In [ ]:

GPT_CONFIG_124M = {
    "vocab_size": 50257,    # Vocabulary size
    "context_length": 256, # Context length
    "emb_dim": 768,         # Embedding dimension
    "n_heads": 12,          # Number of attention heads
    "n_layers": 12,         # Number of layers
    "drop_rate": 0.1,       # Dropout rate
    "qkv_bias": False       # Query-Key-Value bias
}


In [ ]:
class LayerNorm(nn.Module):
  def __init__(self,emb_dim):
    super().__init__()
    self.eps=1e-5
    self.scale=nn.Parameter(torch.ones(emb_dim))
    self.shift=nn.Parameter(torch.zeros(emb_dim))
  def forward(self,x):
    mean=x.mean(dim=-1,keepdim=True)
    var=x.var(dim=-1,keepdim=True,unbiased=True)
    norm_x=(x-mean)/torch.sqrt(var+self.eps)
    return self.scale*norm_x+self.shift

In [ ]:
import math

In [ ]:
class GELU(nn.Module):
  def __init__(self):
    super().__init__()
  def forward(self, x):
        return 0.5 * x * (
            1 + torch.tanh(
                math.sqrt(2.0 / math.pi) * (x + 0.044715 * x**3)
            )
        )

In [ ]:
class FeedForward(nn.Module):
  def __init__(self,cfg):
    super().__init__()
    self.layers=nn.Sequential(
        nn.Linear(cfg["emb_dim"],4*cfg["emb_dim"]),
        GELU(),
        nn.Linear(4*cfg["emb_dim"],cfg["emb_dim"])
    )
  def forward(self,x):
    return self.layers(x)

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert (d_out % num_heads == 0), \
            "d_out must be divisible by num_heads"

        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads # Reduce the projection dim to match desired output dim

        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)  # Linear layer to combine head outputs
        self.dropout = nn.Dropout(dropout)
        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_length, context_length),
                       diagonal=1)
        )
    def forward(self, x):
        b, num_tokens, d_in = x.shape

        keys = self.W_key(x) # Shape: (b, num_tokens, d_out)
        queries = self.W_query(x)
        values = self.W_value(x)

        # We implicitly split the matrix by adding a `num_heads` dimension
        # Unroll last dim: (b, num_tokens, d_out) -> (b, num_tokens, num_heads, head_dim)
        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim)
        values = values.view(b, num_tokens, self.num_heads, self.head_dim)
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)

        # Transpose: (b, num_tokens, num_heads, head_dim) -> (b, num_heads, num_tokens, head_dim)
        keys = keys.transpose(1, 2)
        queries = queries.transpose(1, 2)
        values = values.transpose(1, 2)

        # Compute scaled dot-product attention (aka self-attention) with a causal mask
        attn_scores = queries @ keys.transpose(2, 3)  # Dot product for each head

        # Original mask truncated to the number of tokens and converted to boolean
        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]

        # Use the mask to fill attention scores
        attn_scores.masked_fill_(mask_bool, -torch.inf)

        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)

        # Shape: (b, num_tokens, num_heads, head_dim)
        context_vec = (attn_weights @ values).transpose(1, 2)

        # Combine heads, where self.d_out = self.num_heads * self.head_dim
        context_vec = context_vec.contiguous().view(b, num_tokens, self.d_out)
        context_vec = self.out_proj(context_vec) # optional projection

        return context_vec


In [ ]:
class TransformerBlock(nn.Module):
  def __init__(self,cfg):
    super().__init__()
    self.att=MultiHeadAttention(
      d_in=cfg["emb_dim"],
      d_out=cfg["emb_dim"],
      context_length=cfg["context_length"],
      num_heads=cfg["n_heads"],
      dropout=cfg["drop_rate"],
      qkv_bias=cfg["qkv_bias"])
    self.ff=FeedForward(cfg)
    self.norm1=LayerNorm(cfg["emb_dim"])
    self.norm2=LayerNorm(cfg["emb_dim"])
    self.drop_shortcut=nn.Dropout(cfg["drop_rate"])
  def forward(self,x):
    shortcut=x
    x=self.norm1(x)
    x=self.att(x)
    x=self.drop_shortcut(x)
    x+=shortcut
    shortcut=x
    x=self.norm2(x)
    x=self.ff(x)
    x=self.drop_shortcut(x)
    x+=shortcut
    return x

In [ ]:
# Token Embedding
# Positional Embedding
# Input Embedding=Token Embedding + Positional Embedding
# Dropout (Randomly turn off embedding elements to zero with a probability of p)
# Transformer

  ## Layer Normalization(with mean =0 variance =1)
  ## Masked Multi head attention to Generate contect vector embedding using attention weight for no of head
  ## Dropout
  ## Shortcut connection(prevents VGD)
  ## Layer Norm
  ## Feed Forward (expnasion and contraction)
  ## Output dimension is same as input
  ## Dropout
  ## Shortcut Connection
  ## Transformer block Output
# Layer Normalisation
# Output Head(Neural layer (emb_dim*vocab size))
# Output Logits for each word we have vocabsize no of token
# Batch_size *cotext size * vocab size




In [ ]:
class GPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
        self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
        self.drop_emb = nn.Dropout(cfg["drop_rate"])

        self.trf_blocks = nn.Sequential(
            *[TransformerBlock(cfg) for _ in range(cfg["n_layers"])])

        self.final_norm = LayerNorm(cfg["emb_dim"])
        self.out_head = nn.Linear(
            cfg["emb_dim"], cfg["vocab_size"], bias=False
        )

    def forward(self, in_idx):
        batch_size, seq_len = in_idx.shape
        tok_embeds = self.tok_emb(in_idx)
        pos_embeds = self.pos_emb(torch.arange(seq_len, device=in_idx.device))
        x = tok_embeds + pos_embeds  # Shape [batch_size, num_tokens, emb_size]
        x = self.drop_emb(x)
        x = self.trf_blocks(x)
        x = self.final_norm(x)
        logits = self.out_head(x)
        return logits

In [ ]:
torch.manual_seed(123)
model=GPTModel(GPT_CONFIG_124M)
batch=torch.tensor(
    [[6109,3626,6100,345],
     [6109,1110,6622,257]]
)
out=model(batch)
print(f"Batch is :{batch}")
print(f"\nOutput shape:{out.shape}")
print(out)

Batch is :tensor([[6109, 3626, 6100,  345],
        [6109, 1110, 6622,  257]])

Output shape:torch.Size([2, 4, 50257])
tensor([[[-1.7930e-01,  2.8437e-01, -7.6031e-01,  ..., -4.8328e-01,
          -4.2459e-01, -1.7183e-01],
         [-6.2540e-01, -3.7472e-01, -9.6911e-01,  ...,  1.9217e-01,
          -1.3227e+00, -2.7610e-01],
         [ 5.1774e-01,  1.3858e-01,  2.4931e-01,  ...,  3.5055e-01,
          -7.7139e-02, -7.9681e-02],
         [-2.5624e-01, -6.9668e-01, -9.9378e-01,  ..., -4.4322e-02,
           6.1900e-02,  1.3455e-01]],

        [[-2.2345e-01,  1.1610e-01, -9.9719e-01,  ..., -1.5698e-01,
          -4.4743e-01, -2.8685e-02],
         [-8.7201e-01, -3.9409e-01, -1.1089e+00,  ...,  3.3024e-01,
          -9.2300e-02, -9.1270e-05],
         [ 4.5925e-01, -1.4261e-01, -1.2172e-01,  ...,  2.7475e-01,
           5.8875e-02, -9.0157e-02],
         [-6.2152e-01, -4.4847e-01, -4.7587e-01,  ..., -3.6475e-01,
           3.4392e-01, -3.8138e-01]]], grad_fn=<UnsafeViewBackward0>)


In [ ]:
total_params=sum(p.numel() for p in model.parameters())
print(f"total number of parameters:{total_params:,}")

total number of parameters:162,419,712


In [ ]:
def gen_text(model,idx,max_new_tokens,context_size):
  for _ in range(max_new_tokens):
    idx_cond=idx[:,-context_size:]
    with torch.no_grad():
      logits=model(idx_cond)
    logits=logits[:,-1,:]
    probabs=torch.softmax(logits,dim=-1)
    idx_next = torch.multinomial(probabs, num_samples=1)
    idx=torch.cat((idx,idx_next),dim=1)
  return idx


In [ ]:
torch.manual_seed(123)
start_content="hello,I am"
encoded=tokenizer.encode(start_content)
print("Encoded",encoded)
encoded_tensor=torch.tensor(encoded).unsqueeze(0)
print("encoded_tensor_shape",encoded_tensor.shape)

Encoded [31373, 11, 40, 716]
encoded_tensor_shape torch.Size([1, 4])


In [ ]:
model.eval()
out=gen_text(model=model,idx=encoded_tensor,max_new_tokens=10,context_size=GPT_CONFIG_124M["context_length"])
print("Output",out)
print(len(out[0]))

Output tensor([[31373,    11,    40,   716, 24075, 41835, 16958, 12204, 14951, 39328,
         43096,  3922,  3776, 43071]])
14


In [ ]:
decoded_text=tokenizer.decode(out.squeeze(0).tolist())
print(decoded_text)

hello,I am Wisdom Stores whoeverroc Superman wallpaper servingsilies Game stren


In [ ]:
def text_token(text,tokenizer):
  encoded=tokenizer.encode(text,allowed_special={'<|endoftext|>'})
  encoded_tensor=torch.tensor(encoded).unsqueeze(0)
  return encoded_tensor
def token_text(token_ids,tokenizer):
  flat=token_ids.squeeze(0)
  return tokenizer.decode(flat.tolist())
start_content="Every effort moves you"
tokenizer=tiktoken.get_encoding("gpt2")
token_ids=gen_text(
    model=model,
    idx=text_token(start_content,tokenizer),
    max_new_tokens=10,
    context_size=GPT_CONFIG_124M["context_length"]
)
print("Output text\n",token_text(token_ids,tokenizer))

Output text
 Every effort moves you rentingetic wasnم refres RexMeCHicular stren


In [ ]:
inputs=torch.tensor([[16833,3626,6100],
                     [40,1107,588]])
targets=torch.tensor([[3626,6100,345],
                      [1107,588,11311]])


In [ ]:
with torch.no_grad():
  logits=model(inputs)
probabs=torch.softmax(logits,dim=-1)
print(probabs,"\n",probabs.shape)


tensor([[[1.8852e-05, 1.5172e-05, 1.1698e-05,  ..., 2.2408e-05,
          6.9822e-06, 1.8781e-05],
         [9.1619e-06, 1.0067e-05, 7.8848e-06,  ..., 2.9088e-05,
          6.0139e-06, 1.3577e-05],
         [2.9887e-05, 8.8599e-06, 1.5754e-05,  ..., 3.5435e-05,
          1.4104e-05, 1.3535e-05]],

        [[1.2571e-05, 2.0535e-05, 1.4342e-05,  ..., 1.0396e-05,
          3.4776e-05, 1.4245e-05],
         [7.2785e-06, 1.7863e-05, 1.0568e-05,  ..., 2.1211e-05,
          1.1390e-05, 1.5565e-05],
         [2.9499e-05, 3.3594e-05, 4.1009e-05,  ..., 6.5304e-06,
          5.8152e-05, 1.3705e-05]]]) 
 torch.Size([2, 3, 50257])


In [ ]:
token_ids=torch.argmax(probabs,dim=-1,keepdim=True)
print("Token IDs:\n",token_ids)

Token IDs:
 tensor([[[16657],
         [  339],
         [42826]],

        [[49906],
         [29669],
         [41751]]])


In [ ]:
print(f"Targets batch 1:{token_text(targets[0],tokenizer)}")
print(f"Targets batch 1:{token_text(token_ids[0].flatten(),tokenizer)}")

Targets batch 1: effort moves you
Targets batch 1: Armed heNetflix


In [ ]:
import torch.nn as nn

In [ ]:
text_idx=0
target_probab=probabs[text_idx,[0,1,2],targets[text_idx]]
print("Text 1",target_probab)
text_idx=1
target_probabs=probabs[text_idx,[0,1,2],targets[text_idx]]
print("text 2",target_probabs)

Text 1 tensor([7.4514e-05, 3.1054e-05, 1.1567e-05])
text 2 tensor([1.0343e-05, 5.6737e-05, 4.7620e-06])


In [ ]:
log_probabs=torch.log(torch.cat((target_probab,target_probabs)))
print(log_probabs)

tensor([ -9.5045, -10.3798, -11.3674, -11.4792,  -9.7771, -12.2549])


In [ ]:
avg_log=torch.mean(log_probabs)
print(avg_log)

tensor(-10.7938)


In [ ]:
pos_log=avg_log*-1

In [ ]:
logits_flat=logits.flatten(0,1)
targets_flat=targets.flatten()
print(logits_flat.shape,targets_flat.shape)

torch.Size([6, 50257]) torch.Size([6])


In [ ]:
loss=nn.functional.cross_entropy(logits_flat,targets_flat)
print(loss)

tensor(10.7938)


In [ ]:
perplexity=torch.exp(loss)
perplexity

tensor(48717.6914)

In [ ]:
from sklearn.model_selection import train_test_split

# Lets work on real data

In [ ]:
import os
file_path="/content/the-verdict.txt"
with open(file_path,"r") as f:
  text_data=f.read()

In [ ]:
text_data[:99]

'I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no '

In [ ]:
text_data[-100:]

' it for me! The Strouds stand alone, and happen once--but there\'s no exterminating our kind of art."'

In [ ]:
total_char=len(text_data)
total_token=len(tokenizer.encode(text_data))
print(f"Characters:",total_char)
print(f"Tokens:",total_token)


Characters: 20479
Tokens: 5145


In [ ]:
from torch.utils.data import Dataset,DataLoader
class GPTDatsetV1(Dataset):
  def __init__(self,txt,tokenizer,max_length,stride):
    self.input_ids=[]
    self.target_ids=[]
    token_ids=tokenizer.encode(txt,allowed_special={'<|endoftext|>'})
    for i in range(0,len(token_ids)-max_length,stride):
      input_chunk=token_ids[i:i+max_length]
      target_chunk=token_ids[i+1:i+max_length+1]
      self.input_ids.append(torch.tensor(input_chunk))
      self.target_ids.append(torch.tensor(target_chunk))
  def __len__(self):
    return len(self.input_ids)
  def __getitem__(self,idx):
    return self.input_ids[idx],self.target_ids[idx]

In [ ]:
def create_dataloader_v1(txt,batch_size=4,max_length=256,stride=128,shuffle=True,drop_last=True,num_workers=0):
  tokenizer=tiktoken.get_encoding("gpt2")
  dataset=GPTDatsetV1(txt,tokenizer,max_length,stride)
  dataloader=DataLoader(
      dataset,
      batch_size=batch_size,
      shuffle=shuffle,
      drop_last=drop_last,
      num_workers=num_workers
  )
  return dataloader


In [ ]:
train_ratio=0.90
split_idx=int(train_ratio*len(text_data))
train_data=text_data[:split_idx]
val_data=text_data[split_idx:]
torch.manual_seed(123)
train_loader=create_dataloader_v1(
    train_data,
    batch_size=2,
    max_length=GPT_CONFIG_124M["context_length"],
    stride=GPT_CONFIG_124M['context_length'],
    drop_last=True,
    shuffle=True,
    num_workers=0
)
test_loader=create_dataloader_v1(
    val_data,
    batch_size=2,
    max_length=GPT_CONFIG_124M["context_length"],
    stride=GPT_CONFIG_124M['context_length'],
    drop_last=False,
    shuffle=False,
    num_workers=0
)

In [ ]:
print("Train Loader:")
for x,y in train_loader:
  print(x.shape,y.shape)
print("Test Loader:")
for x,y in test_loader:
  print(x.shape,y.shape)

Train Loader:
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
Test Loader:
torch.Size([2, 256]) torch.Size([2, 256])


In [ ]:
class GPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
        self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
        self.drop_emb = nn.Dropout(cfg["drop_rate"])

        self.trf_blocks = nn.Sequential(
            *[TransformerBlock(cfg) for _ in range(cfg["n_layers"])])

        self.final_norm = LayerNorm(cfg["emb_dim"])
        self.out_head = nn.Linear(
            cfg["emb_dim"], cfg["vocab_size"], bias=False
        )

    def forward(self, in_idx):
        batch_size, seq_len = in_idx.shape
        tok_embeds = self.tok_emb(in_idx)
        pos_embeds = self.pos_emb(torch.arange(seq_len, device=in_idx.device))
        x = tok_embeds + pos_embeds  # Shape [batch_size, num_tokens, emb_size]
        x = self.drop_emb(x)
        x = self.trf_blocks(x)
        x = self.final_norm(x)
        logits = self.out_head(x)
        return logits
torch.manual_seed(123)
model=GPTModel(GPT_CONFIG_124M)
model.eval()

GPTModel(
  (tok_emb): Embedding(50257, 768)
  (pos_emb): Embedding(256, 768)
  (drop_emb): Dropout(p=0.1, inplace=False)
  (trf_blocks): Sequential(
    (0): TransformerBlock(
      (att): MultiHeadAttention(
        (W_query): Linear(in_features=768, out_features=768, bias=False)
        (W_key): Linear(in_features=768, out_features=768, bias=False)
        (W_value): Linear(in_features=768, out_features=768, bias=False)
        (out_proj): Linear(in_features=768, out_features=768, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (ff): FeedForward(
        (layers): Sequential(
          (0): Linear(in_features=768, out_features=3072, bias=True)
          (1): GELU()
          (2): Linear(in_features=3072, out_features=768, bias=True)
        )
      )
      (norm1): LayerNorm()
      (norm2): LayerNorm()
      (drop_shortcut): Dropout(p=0.1, inplace=False)
    )
    (1): TransformerBlock(
      (att): MultiHeadAttention(
        (W_query): Linear(in_features

In [ ]:
from tqdm import tqdm

In [ ]:
def cal_loss_batch(input_batch,target_batch,model,device):
  input_batch,target_batch=input_batch.to(device),target_batch.to(device)
  logits=model(input_batch)
  loss=torch.nn.functional.cross_entropy(logits.flatten(0,1),target_batch.flatten())
  return loss
def calc_lossloader(data_loader,model,device,num_batches=None):
  total_loss=0
  if num_batches is None:
    num_batches=len(data_loader)
  else:
    num_batches=min(num_batches,len(data_loader))
  for i,(input_batch,target_batch) in enumerate (data_loader):
    if i<num_batches:
      loss=cal_loss_batch(input_batch,target_batch,model,device)
      total_loss+=loss.item()
    else:
      break
  return total_loss/num_batches



In [ ]:
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [ ]:
model.to(device)
torch.manual_seed(123)
with torch.no_grad():
  train_loss=calc_lossloader(train_loader,model,device)
  val_loss=calc_lossloader(test_loader,model,device)
print(f"Training Loss: {train_loss}")
print(f"Validation Loss:{val_loss}")

Training Loss: 10.98738521999783
Validation Loss:10.980905532836914


# Backpropagation

## Embedding parmas=
50257*768+1024*768=38.4M
Token +positional
## MHA=
3*768*768=1.77M(Query,Key,Value,Weights)
## Output head=
768*768=0.59M
## Total=
2.36M
## Feed Forward=
768*(4*768)+768*(4*768)=4.72M
##Total for 12transformer =
12*(4.72*2.36)=8.52M
## Softmax=
768*50257=162M
## Total param to be optimise=
162M


In [ ]:
def train_model(model,train_loader,val_loader,optimizer,device,num_epochs,eval_freq,eval_iter,start_context,tokenizer):
  train_losses,val_losses,track_token_seen=[],[],[]
  token_seen,global_step=0,-1
  for epoch in range(num_epochs):
    model.train()
    for input_batch,target_batch in train_loader:
      optimizer.zero_grad()
      loss=cal_loss_batch(input_batch,target_batch,model,device)
      loss.backward()
      optimizer.step()
      token_seen+=input_batch.numel()
      global_step+=1
      if global_step % eval_freq==0:
        train_loss,val_loss=evaluate_model(
            model,train_loader,val_loader,device,eval_iter)
        train_losses.append(train_loss)
        val_losses.append(val_loss)
        track_token_seen.append(token_seen)
        print(f"Ep{epoch+1} (Step{global_step:06d}):",
              f"Train Loss {train_loss:3f},Val loss {val_loss:3f}")
    generate_sample(model,tokenizer,device,start_context)
  return train_losses,val_losses,track_token_seen




In [ ]:
def evaluate_model(model,train_loader,val_loader,device,eval_iter):
  model.eval()
  with torch.no_grad():
    train_loss=calc_lossloader(train_loader,model,device,num_batches=eval_iter)
    val_loss=calc_lossloader(val_loader,model,device,num_batches=eval_iter)
  model.train()
  return train_loss,val_loss

In [ ]:
def generate_sample(model,tokenizer,device,start_context):
  model.eval()
  context_size=model.pos_emb.weight.shape[0]
  encoded=text_token(start_context,tokenizer).to(device)
  with torch.no_grad():
    token_ids=gen_text(model=model,idx=encoded,max_new_tokens=50,context_size=context_size)
  decoded_text=token_text(token_ids,tokenizer)
  print(decoded_text.replace('\n'," "))
  model.train()


In [ ]:
import time
start_time=time.time()
torch.manual_seed(123)
model=GPTModel(GPT_CONFIG_124M)
model.to(device)
optimizer=torch.optim.Adam(model.parameters(),lr=0.0004,weight_decay=0.1)
num_epochs=50
train_losses,val_losses,token_seen=train_model(
    model,train_loader=train_loader,val_loader=test_loader,optimizer=optimizer,device=device,num_epochs=num_epochs,eval_freq=5,eval_iter=5,start_context="Every effort moves you",tokenizer=tokenizer
)

end_time=time.time()
print(f"Training completed in {(end_time-start_time)/60}")

Ep1 (Step000000): Train Loss 10.678507,Val loss 10.694647
Ep1 (Step000005): Train Loss 9.481247,Val loss 9.689301
Every effort moves you Maggieorb Expl hars neur PepsiAmericanCostBarspoken -= upgradesGeorgeawarurup 465OVER includes RIS Cameronorthy Hes bends gluc evenlysv Raven erasesweet that tv crisp FY frying maneuversField drive repaylibrary suited 388urdyön Vacc leaflets DisTankkat crackingura
Ep2 (Step000010): Train Loss 8.832129,Val loss 9.061780
Ep2 (Step000015): Train Loss 8.407733,Val loss 8.627594
Every effort moves you traveler Berg ApacheVis lecturesiateites,nesium Conj laughelfthmeta Squirrel Dramauration the Reevesabsorconstructuppetitz to Dud unofficial SUR Man Introduction marrow instanceazor, Advoc Mash impoverPool mindfulness censored had licens mac AdrenFlags  Seeing challengers warriorsHongpredagement
Ep3 (Step000020): Train Loss 8.051849,Val loss 8.223683
Ep3 (Step000025): Train Loss 7.658643,Val loss 7.874866
Every effort moves you evade managedantlehift derail a